In [ ]:
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ==============================================================================
# 0. CONFIGURACIÓN DEL EXPERIMENTO ÚNICO (Para OLS, RF, XGBoost, etc.)
# ==============================================================================
experiment_name = "/modelos-hipervalorizacion-bogota"
mlflow.set_experiment(experiment_name)

print("--- 1. CARGA Y PREPARACIÓN DE DATOS ---")
matriz = "/Volumes/workspace/default/datos_bogota/matriz_barrios_unificada.csv"
df = pd.read_csv(matriz)

# Separar datos supervisados vs inferencia
df_supervised = df[df['v_ref_2026'].notnull()].copy()
df_inference = df[df['v_ref_2026'].isnull()].copy()

# Preparar Matriz X e y
y_sup = df_supervised['v_ref_2026']
cols_a_excluir = ['barrio', 'geometry', 'v_ref_2015', 'v_ref_2026', 'num_listings_airbnb', 'area_km2']
X_sup = df_supervised.drop(columns=[col for col in cols_a_excluir if col in df_supervised.columns])
X_sup = pd.get_dummies(X_sup, columns=['localidad'], drop_first=True, dtype=int)

# Train / Test Split
X_train, X_test, y_train, y_test = train_test_split(X_sup, y_sup, test_size=0.2, random_state=42)

# Parámetros compartidos
params = {
    "n_estimators": 500,
    "max_depth": 15,
    "min_samples_split": 15,
    "min_samples_leaf": 5,
    "max_features": 1.0,
    "random_state": 42
}

# ==============================================================================
# RUN 1: RANDOM FOREST SIN LOGARITMO (Precios Brutos)
# ==============================================================================
with mlflow.start_run(run_name="RandomForest_Sin_Log"):
    mlflow.log_params(params)
    mlflow.log_param("target_transform", "Ninguna")

    rf_bruto = RandomForestRegressor(**params, n_jobs=-1)
    rf_bruto.fit(X_train, y_train)

    y_pred_bruto = rf_bruto.predict(X_test)
    
    mae_bruto = mean_absolute_error(y_test, y_pred_bruto)
    rmse_bruto = np.sqrt(mean_squared_error(y_test, y_pred_bruto))
    r2_bruto = r2_score(y_test, y_pred_bruto)

    y_pred_train = rf_bruto.predict(X_train)
    
    mae_train = mean_absolute_error(y_train, y_pred_train)
    r2_train = r2_score(y_train, y_pred_train)
    
    mlflow.log_metric("mae_train_pesos", mae_train)
    mlflow.log_metric("mae_test_pesos", mae_bruto) 
    mlflow.log_metric("r2_train", r2_train)
    mlflow.log_metric("r2_test", r2_bruto)     
    mlflow.log_metric("rmse_test_pesos", rmse_bruto) # Faltaba registrar esta métrica
    mlflow.sklearn.log_model(rf_bruto, "modelo_rf_bruto") # Faltaba guardar el modelo

# ==============================================================================
# RUN 2: RANDOM FOREST CON LOGARITMO + INFERENCIA FINAL
# ==============================================================================
with mlflow.start_run(run_name="RandomForest_Con_Log"):
    mlflow.log_params(params)
    mlflow.log_param("target_transform", "Logaritmo Natural")

    # Transformación
    y_train_log = np.log(y_train)
    
    rf_log = RandomForestRegressor(**params, n_jobs=-1)
    rf_log.fit(X_train, y_train_log)

    # Predicción y reversión (np.exp) para evaluar en escala real
    y_pred_log = rf_log.predict(X_test)
    y_pred_revertido = np.exp(y_pred_log)
    
    mae_log_rev = mean_absolute_error(y_test, y_pred_revertido)
    rmse_log_rev = np.sqrt(mean_squared_error(y_test, y_pred_revertido))
    r2_log_rev = r2_score(y_test, y_pred_revertido)

    y_pred_train_log = rf_log.predict(X_train)
    y_pred_train_revertido = np.exp(y_pred_train_log)

    mae_train_rev = mean_absolute_error(y_train, y_pred_train_revertido)
    r2_train_rev = r2_score(y_train, y_pred_train_revertido)

    mlflow.log_metric("mae_train_pesos", mae_train_rev)
    mlflow.log_metric("mae_test_pesos", mae_log_rev)
    mlflow.log_metric("r2_train", r2_train_rev)
    mlflow.log_metric("r2_test", r2_log_rev)
    mlflow.log_metric("rmse_test_pesos", rmse_log_rev)
    mlflow.sklearn.log_model(rf_log, "modelo_rf_log")